In [ ]:
import pickle
import os
import pandas as pd
import ast
from collections import defaultdict


EVENTS_PATH = r"/workspace/qgi\qgi-4.0\data\events"   # r"/workspaces/qgi-4.0/data/events"


class Country:
    def __init__(self):
        self.name      = ""
        self.id        = ""
        self.patterns  = []
        self.sdcis     = []
        self.agg_sdcis = []
        self.events    = list[dict]

class Pattern:
    def __init__(self):
        self.country_a_id    = ""
        self.country_a       = ""
        self.country_b_id    = ""
        self.country_b       = ""
        self.pattern_length  = ""
        self.start_year_a_fk = ""
        self.start_year_b_fk = ""
        self.indexes         = ""
        self.avg_corr        = ""
        self.orgs            = ""
        self.power           = ""
        self.sectors         = []

class PatternsdciCouple:
    def __init__(self):
        self.pi_tuple = ()

class SDCI:
    def __init__(self):
        self.unique_id           = ""
        self.org_fk              = ""
        self.main_name_fk        = ""
        self.index_name_fk       = ""
        self.version_fk          = 0
        self.country_a_id_fk     = 0
        self.country_b_id_fk     = 0
        self.pattern_length_fk   = 0
        self.correlation         = 0.0
        self.start_year_a_fk     = 0
        self.index_values_a      = []
        self.start_year_b_fk     = 0
        self.index_values_b      = []
        self.creation_date       = ""
        self.sector              = ""


## General Utils
def save_class_as_pkl(class_var, name: str):
    with open(name, "wb") as f:
        pickle.dump(class_var, f)

def read_pkl_as_class(pkl_path: str):
    with open(pkl_path, "rb") as f:
        return pickle.load(f)

def count_event_types(countries):
    event_counts = defaultdict(int)
    for country in countries:
        for event in country.events:
            event_type = event['event_type']
            event_counts[event_type] += 1
    return dict(event_counts)

def rank_event_types(countries):
    event_counts = count_event_types(countries)
    sorted_events = sorted(event_counts.items(), key=lambda x: x[1], reverse=True)
    return sorted_events

## Recipe functions
def read_events_data(country_name):
    try:
        path = os.path.join(EVENTS_PATH, f"{country_name}.txt")
        with open(path, "r") as f:
            events = f.readlines()
        parsed_events = []
        for event in events:
            parts = event.strip().split(" | ")
            if len(parts) == 4:
                _, year, event_type, description = parts
                parsed_events.append({
                    "year": int(year),
                    "event_type": event_type,
                    "description": description
                })
        return parsed_events
    except Exception as e: #FileNotFoundError:
        if "No such file or directory" not in str(e):
            print(str(e))
            print(path)
        return []

# Identify patterns leading to an event
def get_patterns_leading_to_event(loaded_countries: list[Country], event_type, margin=2):
    event_patterns = []
    
    for country in loaded_countries:
        # Get event years for this country
        event_years = [e["year"] for e in country.events if e["event_type"] == event_type]
        
        # Check each pattern for this country
        for pattern in country.patterns:
            # Calculate pattern end years
            end_year_a = pattern.start_year_a_fk + pattern.pattern_length
            end_year_b = pattern.start_year_b_fk + pattern.pattern_length
            
            # Check if either country has an event within the margin around the end year
            has_event_a = any(abs(end_year_a - ey) <= margin for ey in event_years)
            
            # Get country B
            country_b = next((c for c in loaded_countries if c.id == pattern.country_b_id), None)
            if country_b:
                country_b_event_years = [e["year"] for e in country_b.events if e["event_type"] == event_type]
                has_event_b = any(abs(end_year_b - ey) <= margin for ey in country_b_event_years)
                
                if has_event_a and has_event_b:
                    event_patterns.append(pattern)
    
    return event_patterns

def create_event_recipe(patterns: list, sdcis_df: pd.DataFrame, take_common_sdcis: bool = False):
    recipe = defaultdict(list)
    all_sdcis = []

    # Collect all SDCIs from all patterns
    for pattern in patterns:
        sdcis = get_sdcis_for_pattern(pattern, sdcis_df)
        all_sdcis.append(sdcis)

    if take_common_sdcis:
        # Find SDCIs common to all patterns
        common_sdcis = None
        for sdcis in all_sdcis:
            if common_sdcis is None:
                common_sdcis = set(sdcis[['index_name_fk', 'sector']].apply(tuple, axis=1))
            else:
                common_sdcis.intersection_update(set(sdcis[['index_name_fk', 'sector']].apply(tuple, axis=1)))
        
        # If no common SDCIs found, return empty recipe
        if not common_sdcis:
            return {}
        
        # Filter SDCIs to only include common ones
        for sdcis in all_sdcis:
            for _, sdci in sdcis.iterrows():
                if (sdci['index_name_fk'], sdci['sector']) in common_sdcis:
                    recipe[(sdci['index_name_fk'], sdci['sector'])].append(sdci['correlation'])
    else:
        # Include all SDCIs from all patterns
        for sdcis in all_sdcis:
            for _, sdci in sdcis.iterrows():
                recipe[(sdci['index_name_fk'], sdci['sector'])].append(sdci['correlation'])

    # Calculate average correlations
    return {k: sum(v)/len(v) for k, v in recipe.items()}

def load_sdcis_from_parquet(folder_path):
    sdcis = []
    for file in os.listdir(folder_path):
        if file.endswith('.parquet'):
            file_path = os.path.join(folder_path, file)
            sdcis.append(pd.read_parquet(file_path))
    return pd.concat(sdcis, ignore_index=True)

# Function to get SDCIs for a pattern
def get_sdcis_for_pattern(pattern: Pattern, sdcis_df):
    return sdcis_df.query(
        f"country_a_id_fk == {pattern.country_a_id} and "
        f"country_b_id_fk == {pattern.country_b_id} and "
        f"pattern_length_fk == {pattern.pattern_length} and "
        f"start_year_a_fk == {pattern.start_year_a_fk} and "
        f"start_year_b_fk == {pattern.start_year_b_fk}"
    )


## Loading SDCIs parquet and countries class pkl
sdcis_df = load_sdcis_from_parquet(r"/workspace/qgi\qgi-4.0\data\countries_sdci_parquet")
countries = read_pkl_as_class("all.pkl")

## Loading country events
for country in countries:
    country.events = read_events_data(country.name)


'charmap' codec can't decode byte 0x81 in position 2416: character maps to <undefined>
/workspace/qgi\qgi-4.0\data\events\New Zealand.txt
'charmap' codec can't decode byte 0x90 in position 117: character maps to <undefined>
/workspace/qgi\qgi-4.0\data\events\Vietnam.txt


In [68]:
EVENT = "Economic Growth"
military_conflict_patterns = get_patterns_leading_to_event(countries, EVENT, 0)
military_conflict_recipe = create_event_recipe(military_conflict_patterns, sdcis_df)

print(EVENT + " Recipe:")
for (index, sector), avg_corr in military_conflict_recipe.items():
    print(f"{index} ({sector}): {avg_corr:.2f}")

Economic Growth Recipe:
Net Trade (Unknown): 0.97
Imports (Unknown): 0.93
E2: Economic Inequality (Human Rights Development): 0.97
Safety & Security (Unknown): 0.94
Primary Energy Supply (Unknown): 0.96
External debt stocks (% of GNI) (Unknown): 0.97
Infant Mortality Rates (Unknown): 0.95
Suicide Rates (Unknown): 0.94
GDP per capita growth (annual %) (Unknown): 0.98
Military Expenditure - Share of GDP (%) (Military Sector): 0.95
Net Trade (Trade Sector): 0.96
C2: Factionalized Elites (Social Sector): 0.93
E3: Human Flight and Brain Drain (Human Rights Development): 0.95
P2: Public Services (Political State Sector): 0.96
S1: Demographic Pressures (Social Sector): 0.96
Total (Finance Economy Sector): 0.95
E1: Economy (Finance Economy Sector): 0.97
Military Expenditure - Per Capita (Military Sector): 0.96
C1: Security Apparatus (Social Sector): 0.97
C3: Group Grievance (Social Sector): 0.95
X1: External Intervention (Sovereignty): 0.96
P3: Human Rights (Human Rights Development): 0.96
S2:

In [63]:
event_frequencies = rank_event_types(countries)
for event_type, count in event_frequencies:
    print(f"{event_type}: {count}")

Infrastructure Development: 265
Economic Growth: 235
Education Reform: 159
Presidential Election: 135
Economic Reforms: 128
Environmental Initiatives: 112
Environmental Policy: 106
International Relations: 104
Cultural Heritage Preservation: 85
Infrastructure Projects: 82
Anti-Corruption Measures: 81
Economic Recovery: 79
Educational Reform: 78
Technological Advancement: 78
Education Reforms: 77
Environmental Legislation: 77
Environmental Conservation: 75
Economic Diversification: 74
Economic Challenges: 64
Diplomatic Relations: 63
Political Reforms: 61
Economic Reform: 61
Judicial Reforms: 56
Environmental Initiative: 56
Economic Crisis: 54
Natural Disaster: 54
Technological Innovation: 53
Parliamentary Elections: 51
Social Reforms: 50
Renewable Energy Projects: 47
Economic Recession: 47
Educational Reforms: 46
Humanitarian Crisis: 46
Infrastructure Expansion: 46
Environmental Protection: 46
Renewable Energy Initiatives: 43
International Sporting Event: 42
Social Programs: 41
Educatio